# 05 · Topic patterns and header-based routing

**Level:** beginner/intermediate · **Time:** 35–45 minutes · **Prerequisite:** notebook 04.

Route order events by topic and then by metadata headers. Choose topic routing when a compact hierarchy
such as `order.created.eu` describes the route; choose headers when independent attributes are clearer.

Topic keys contain dot-separated words. In bindings, `*` matches exactly one word and `#` matches zero
or more words. Header exchanges compare AMQP message headers and ignore routing keys for matching.


In [ ]:
from pathlib import Path
import sys
import json
import time
import pika

ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / "course" / "rabbit.py").is_file()), None)
assert ROOT is not None, "Open Jupyter from the RabbitMQ course folder."
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from course.rabbit import connect, unique_name, wait_for_message


## 1. Predict this topic-routing table

| Published key | `order.*.eu` | `order.#` | `#.eu` |
|---|---|---|---|
| `order.created.eu` | yes | yes | yes |
| `order.paid.us` | no | yes | no |
| `order.eu` | no | yes | yes |
| `inventory.low.eu` | no | no | yes |

The short key `order.eu` is the useful edge case: `*` cannot match zero words.


In [ ]:
topic_exchange = unique_name("events.topic")
topic_bindings = {
    unique_name("eu.order"): "order.*.eu",
    unique_name("all.order"): "order.#",
    unique_name("all.eu"): "#.eu",
}
keys = ["order.created.eu", "order.paid.us", "order.eu", "inventory.low.eu"]
with connect() as connection:
    channel = connection.channel()
    channel.exchange_declare(exchange=topic_exchange, exchange_type="topic")
    for queue, pattern in topic_bindings.items():
        channel.queue_declare(queue=queue)
        channel.queue_bind(exchange=topic_exchange, queue=queue, routing_key=pattern)
    for key in keys:
        channel.basic_publish(exchange=topic_exchange, routing_key=key, body=key.encode())
print("Published:", keys)


## 2. Check your predictions

Read the expected number of messages per queue, acknowledge them, and confirm no extras remain.
This experiment tests zero-word matching as well as a one-word match.


In [ ]:
expected = {
    "order.*.eu": ["order.created.eu"],
    "order.#": ["order.created.eu", "order.paid.us", "order.eu"],
    "#.eu": ["order.created.eu", "order.eu", "inventory.low.eu"],
}
with connect() as connection:
    channel = connection.channel()
    for queue, pattern in topic_bindings.items():
        seen = []
        for _ in expected[pattern]:
            method, _, body = wait_for_message(channel, queue)
            seen.append(body.decode())
            channel.basic_ack(method.delivery_tag)
        assert seen == expected[pattern], (pattern, seen)
        assert channel.basic_get(queue=queue, auto_ack=False)[0] is None
        print(pattern, "→", seen)


## 3. Route on multiple independent headers

Create two subscriptions for `format=pdf` and `region=eu`. `x-match=all` requires both attributes;
`x-match=any` requires at least one. These are AMQP properties, not fields hidden inside JSON bodies.
Use consistent header types: the string `"1"` and integer `1` are different typed values.


In [ ]:
headers_exchange = unique_name("documents.headers")
headers_queues = {mode: unique_name(f"headers.{mode}") for mode in ["all", "any"]}
documents = [
    {"id": "A", "format": "pdf", "region": "eu"},
    {"id": "B", "format": "pdf", "region": "us"},
    {"id": "C", "format": "csv", "region": "eu"},
    {"id": "D", "format": "csv", "region": "us"},
]
with connect() as connection:
    channel = connection.channel()
    channel.exchange_declare(exchange=headers_exchange, exchange_type="headers")
    for mode, queue in headers_queues.items():
        channel.queue_declare(queue=queue)
        channel.queue_bind(exchange=headers_exchange, queue=queue,
                           arguments={"x-match": mode, "format": "pdf", "region": "eu"})
    for document in documents:
        channel.basic_publish(
            exchange=headers_exchange, routing_key="not.used.for.header.matching",
            body=document["id"].encode(),
            properties=pika.BasicProperties(headers={"format": document["format"],
                                                     "region": document["region"]}),
        )
print("Published four documents.")


## 4. Verify all versus any

Predict which document IDs arrive in each queue. Document D intentionally matches neither subscription
and is not retained by this topology. Reliable unroutable-message detection is addressed in notebook 07.


In [ ]:
with connect() as connection:
    channel = connection.channel()
    for mode, ids in {"all": ["A"], "any": ["A", "B", "C"]}.items():
        seen = []
        for _ in ids:
            method, properties, body = wait_for_message(channel, headers_queues[mode])
            seen.append(body.decode())
            channel.basic_ack(method.delivery_tag)
        assert seen == ids
        assert channel.basic_get(queue=headers_queues[mode], auto_ack=False)[0] is None
        print(mode, "→", seen)


## 5. Exercise: overlapping topic bindings on one queue

Create one queue bound to both `order.#` and `#.eu`. Publish `order.created.eu` once. Will two matching
bindings cause two copies in this **same queue**? Write your prediction, then run the solution.


In [ ]:
overlap_queue = unique_name("overlap")
with connect() as connection:
    channel = connection.channel()
    channel.queue_declare(queue=overlap_queue)
    for pattern in ["order.#", "#.eu"]:
        channel.queue_bind(exchange=topic_exchange, queue=overlap_queue, routing_key=pattern)
    channel.basic_publish(exchange=topic_exchange, routing_key="order.created.eu", body=b"one publication")
    method, _, body = wait_for_message(channel, overlap_queue)
    channel.basic_ack(method.delivery_tag)
    assert body == b"one publication"
    assert channel.basic_get(queue=overlap_queue, auto_ack=False)[0] is None
    print("One queue received one copy, despite two matching bindings.")


## 6. Cleanup and design discussion

The exercise also routed a copy to each previously bound topic queue; cleanup removes those lab messages.
Topic hierarchies are contracts: agree on word order and meaning before independent services adopt them.
Headers provide flexible attribute matching but still require a shared schema and consistent types.


In [ ]:
with connect() as connection:
    channel = connection.channel()
    for queue in [*topic_bindings, *headers_queues.values(), overlap_queue]:
        channel.queue_delete(queue=queue)
    for exchange in [topic_exchange, headers_exchange]:
        channel.exchange_delete(exchange=exchange)
print("Topic and header resources removed.")


**Troubleshooting:** `*` is not a substring wildcard: `order.create*` does not mean every word starting
with `create`. Make sure header metadata is in `BasicProperties(headers=...)`; matching does not inspect
the body. Re-declaring an exchange with another type closes the channel; use the distinct names above.

Official references: [topic tutorial](https://www.rabbitmq.com/tutorials/tutorial-five-python) and
[exchange types](https://www.rabbitmq.com/docs/exchanges).
